# Evolutionary Accessibility Heatmaps (All Cell Types)

Reads `peak_selection/regions_all.parquet` (produced by `13a_peak_selection.ipynb`)
and generates per-cell-type blocked heatmaps and BED files.

**No peak selection or statistics here** — this notebook is purely visualization.
Re-run whenever you want to adjust the heatmap style without redoing the selection.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.evolutionary_heatmap import (
    sanitize_name,
    load_pseudobulk_matrices,
    build_heatmap_matrix,
    plot_evolutionary_heatmap,
    export_beds,
)

print("Imports OK")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE       = "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
CONS_DIR   = f"{BASE}/cross_species_consensus_v3"
DESEQ_DIR  = f"{CONS_DIR}/13_deseq2_R_pseudobulk"
ANNO_PATH  = f"{CONS_DIR}/07_master_annotation/master_annotation_corrected.tsv"
FRAG_DIR   = f"{CONS_DIR}/12_fragment_matrices"

SPECIES = ["Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]

# ── Heatmap settings ───────────────────────────────────────────────────────────
MAX_PER_BLOCK      = 50       # top-N peaks shown per module block
ACCESSIBILITY_VIEW = "donor_region"  # "species" | "donor" | "donor_region"
ROW_ZSCORE         = False

# ── Input / output ─────────────────────────────────────────────────────────────
IN_REGIONS   = Path(DESEQ_DIR) / "peak_selection" / "regions_all.parquet"
OUT_HEATMAPS = Path(DESEQ_DIR) / "heatmaps"
OUT_BEDS     = Path(DESEQ_DIR) / "region_beds"

OUT_HEATMAPS.mkdir(parents=True, exist_ok=True)
OUT_BEDS.mkdir(parents=True, exist_ok=True)

print(f"Reading regions from : {IN_REGIONS}")
print(f"Heatmaps             → {OUT_HEATMAPS}")
print(f"BED files            → {OUT_BEDS}")

In [ ]:
print("Loading regions_all...")
regions_all_df = pd.read_parquet(IN_REGIONS)
cell_types = regions_all_df["cell_type"].drop_duplicates().tolist()
print(f"  {len(cell_types)} cell types, {len(regions_all_df):,} peak-module entries")

print("\nLoading master annotation...")
anno = pd.read_csv(ANNO_PATH, sep="\t", low_memory=False).set_index("peak_id")
orth_cols = {sp: f"{sp}_orth" for sp in SPECIES}
print(f"  {anno.shape[0]:,} peaks")

In [ ]:
# ── Main loop: heatmap + BED export ───────────────────────────────────────────

title_mode_map = {
    "species":      "species-mean",
    "donor":        "donor",
    "donor_region": "donor+region",
}
title_mode = title_mode_map[ACCESSIBILITY_VIEW]

for cell_type in cell_types:
    print(f"\n{'='*65}")
    print(f"  {cell_type}")
    print(f"{'='*65}")

    acc_species_df, acc_donor_df, acc_donor_region_df = load_pseudobulk_matrices(
        cell_type, SPECIES, FRAG_DIR
    )
    if acc_species_df.empty:
        print(f"  [SKIP] no pseudobulk data found")
        continue

    if ACCESSIBILITY_VIEW == "species":
        acc_df = acc_species_df
    elif ACCESSIBILITY_VIEW == "donor":
        acc_df = acc_donor_df
    else:
        acc_df = acc_donor_region_df

    # Reconstruct regions dicts from parquet (preserving module order)
    ct_df = regions_all_df[regions_all_df["cell_type"] == cell_type].copy()
    module_order = ct_df["module"].drop_duplicates().tolist()
    regions     = {}
    regions_all = {}
    for mod in module_order:
        peak_ids = (ct_df[ct_df["module"] == mod]
                    .sort_values("rank")["peak_id"].tolist())
        regions[mod]     = peak_ids[:MAX_PER_BLOCK]
        regions_all[mod] = peak_ids

    # Build heatmap matrix
    heatmap_plot_df, block_labels, block_sizes = build_heatmap_matrix(
        regions=regions,
        acc_df=acc_df,
        anno=anno,
        orth_cols=orth_cols,
        apply_row_zscore=ROW_ZSCORE,
    )
    print(f"\n  Heatmap matrix: {heatmap_plot_df.shape[0]} peaks × "
          f"{heatmap_plot_df.shape[1]} columns")

    # Plot heatmap
    ct_heatmap_dir = OUT_HEATMAPS / sanitize_name(cell_type)
    ct_heatmap_dir.mkdir(exist_ok=True)
    plot_evolutionary_heatmap(
        heatmap_plot_df=heatmap_plot_df,
        block_labels=block_labels,
        block_sizes=block_sizes,
        cell_type=cell_type,
        title_mode=title_mode,
        out_dir=ct_heatmap_dir,
        apply_row_zscore=ROW_ZSCORE,
    )

    # Export BED files
    ct_bed_dir = OUT_BEDS / sanitize_name(cell_type)
    ct_bed_dir.mkdir(exist_ok=True)
    export_beds(
        regions_all=regions_all,
        anno=anno,
        out_dir=ct_bed_dir,
        cell_type=cell_type,
        coord_system="Human",
    )

print(f"\n{'='*65}")
print(f"Done. Processed {len(cell_types)} cell types.")